In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Compară setările Transpiler-ului

*Estimare de utilizare: sub un minut pe un procesor Eagle r3 (NOTĂ: Aceasta este doar o estimare. Timpul tău de execuție poate varia.)*

## Context
Pentru a asigura rezultate mai rapide și mai eficiente, începând cu 1 martie 2024, circuitele și observabilele trebuie transformate pentru a folosi doar instrucțiunile suportate de QPU (unitatea de procesare cuantică) înainte de a fi trimise la primitivele Qiskit Runtime. Le numim circuite și observabile *instruction set architecture* (ISA). O metodă comună pentru a face acest lucru este utilizarea funcției `generate_preset_pass_manager` a Transpiler-ului. Totuși, ai putea alege să urmezi un proces mai manual.

De exemplu, ai putea dori să țintești un subset specific de Qubiți pe un anumit dispozitiv. Acest ghid testează performanța diferitelor setări ale Transpiler-ului, parcurgând întregul proces de creare, transpilare și trimitere a circuitelor.

## Cerințe
Înainte de a începe, asigură-te că ai instalat următoarele:

* Qiskit SDK v1.2 sau mai recent, cu suport pentru [vizualizare](https://docs.quantum.ibm.com/api/qiskit/visualization)
* Qiskit Runtime v0.28 sau mai recent (`pip install qiskit-ibm-runtime`)

## Configurare

In [ ]:
# Create circuit to test transpiler on
from qiskit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.circuit.library import GroverOperator, Diagonal

# Use Statevector object to calculate the ideal output
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram
from qiskit.transpiler import PassManager

from qiskit.circuit.library import XGate
from qiskit.quantum_info import hellinger_fidelity

# Qiskit Runtime
from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    Batch,
    SamplerV2 as Sampler,
)
from qiskit_ibm_runtime.transpiler.passes.scheduling import (
    ASAPScheduleAnalysis,
    PadDynamicalDecoupling,
)

## Pasul 1: Mapează intrările clasice la o problemă cuantică
Creează un Circuit mic pe care Transpiler-ul să încerce să îl optimizeze. Acest exemplu creează un Circuit care execută algoritmul lui Grover cu un oracol care marchează starea `111`. Apoi, simulează distribuția ideală (ce te-ai aștepta să măsori dacă ai rula acest Circuit pe un calculator cuantic perfect de un număr infinit de ori) pentru comparație ulterioară.

In [ ]:
# To run on hardware, select the backend with the fewest number of jobs in the queue
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)
backend.name

'ibm_brisbanse'

In [29]:
oracle = Diagonal([1] * 7 + [-1])
qc = QuantumCircuit(3)
qc.h([0, 1, 2])
qc = qc.compose(GroverOperator(oracle))

qc.draw(output="mpl", style="iqp")

<Image src="../docs/images/guides/circuit-transpilation-settings/extracted-outputs/7e7944c5-68ac-40cf-a0eb-5f4a44d53931-0.avif" alt="Output of the previous code cell" />

In [30]:
ideal_distribution = Statevector.from_instruction(qc).probabilities_dict()

plot_histogram(ideal_distribution)

<Image src="../docs/images/guides/circuit-transpilation-settings/extracted-outputs/761afe09-b669-453f-8363-55070d6c8f57-0.avif" alt="Output of the previous code cell" />

## Step 2: Optimize problem for quantum hardware execution

Next, transpile the circuits for the QPU. You will compare the performance of the transpiler with `optimization_level` set to `0` (lowest) against `3` (highest). The lowest optimization level does the bare minimum needed to get the circuit running on the device; it maps the circuit qubits to the device qubits and adds swap gates to allow all two-qubit operations. The highest optimization level is much smarter and uses lots of tricks to reduce the overall gate count. Since multi-qubit gates have high error rates and qubits decohere over time, the shorter circuits should give better results.

The following cell transpiles `qc` for both values of `optimization_level`, prints the number of two-qubit gates, and adds the transpiled circuits to a list. Some of the transpiler's algorithms are randomized, so it sets a seed for reproducibility.

In [31]:
# Need to add measurements to the circuit
qc.measure_all()

# Find the correct two-qubit gate
twoQ_gates = set(["ecr", "cz", "cx"])
for gate in backend.basis_gates:
    if gate in twoQ_gates:
        twoQ_gate = gate

circuits = []
for optimization_level in [0, 3]:
    pm = generate_preset_pass_manager(
        optimization_level, backend=backend, seed_transpiler=0
    )
    t_qc = pm.run(qc)
    print(
        f"Two-qubit gates (optimization_level={optimization_level}): ",
        t_qc.count_ops()[twoQ_gate],
    )
    circuits.append(t_qc)

Two-qubit gates (optimization_level=0):  21
Two-qubit gates (optimization_level=3):  14


![Rezultatul celulei de cod anterioare](../docs/images/guides/circuit-transpilation-settings/extracted-outputs/761afe09-b669-453f-8363-55070d6c8f57-0.avif)

## Pasul 2: Optimizează problema pentru execuția pe hardware cuantic
Apoi, transpilează circuitele pentru QPU. Vei compara performanța Transpiler-ului cu `optimization_level` setat la `0` (cel mai scăzut) față de `3` (cel mai ridicat). Nivelul de optimizare cel mai scăzut face minimul necesar pentru a rula Circuit-ul pe dispozitiv: mapează Qubiții circuitului la Qubiții dispozitivului și adaugă Gate-uri swap pentru a permite toate operațiile pe doi Qubiți. Nivelul de optimizare cel mai ridicat este mult mai inteligent și folosește numeroase trucuri pentru a reduce numărul total de Gate-uri. Deoarece Gate-urile multi-Qubit au rate de eroare ridicate și Qubiții decoerează în timp, circuitele mai scurte ar trebui să ofere rezultate mai bune.

Celula următoare transpilează `qc` pentru ambele valori ale `optimization_level`, afișează numărul de Gate-uri cu doi Qubiți și adaugă circuitele transpilate într-o listă. Unii algoritmi ai Transpiler-ului sunt randomizați, deci se setează un seed pentru reproductibilitate.

In [ ]:
# Get gate durations so the transpiler knows how long each operation takes
durations = backend.target.durations()

# This is the sequence we'll apply to idling qubits
dd_sequence = [XGate(), XGate()]

# Run scheduling and dynamic decoupling passes on circuit
pm = PassManager(
    [
        ASAPScheduleAnalysis(durations),
        PadDynamicalDecoupling(durations, dd_sequence),
    ]
)
circ_dd = pm.run(circuits[1])

# Add this new circuit to our list
circuits.append(circ_dd)

In [33]:
circ_dd.draw(output="mpl", style="iqp", idle_wires=False)

<Image src="../docs/images/guides/circuit-transpilation-settings/extracted-outputs/4ada6498-b9d7-4d88-b8a9-ef1dc0a85bf7-0.avif" alt="Output of the previous code cell" />

Deoarece CNOT-urile au de obicei o rată de eroare ridicată, Circuit-ul transpilat cu `optimization_level=3` ar trebui să performeze mult mai bine.

O altă modalitate de a îmbunătăți performanța este prin [dynamic decoupling](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.PadDynamicalDecoupling), aplicând o secvență de Gate-uri pe Qubiții inactivi. Aceasta anulează unele interacțiuni nedorite cu mediul. Celula următoare adaugă dynamic decoupling la Circuit-ul transpilat cu `optimization_level=3` și îl adaugă în listă.

In [34]:
with Batch(backend=backend):
    sampler = Sampler()
    job = sampler.run(
        [(circuit) for circuit in circuits],  # sample all three circuits
        shots=8000,
    )
    result = job.result()

## Step 4: Post-process and return result in desired classical format

Finally, plot the results from the device runs against the ideal distribution. You can see the results with `optimization_level=3` are closer to the ideal distribution due to the lower gate count, and `optimization_level=3 + dd` is even closer due to the dynamic decoupling.

In [35]:
binary_prob = [
    {
        k: v / res.data.meas.num_shots
        for k, v in res.data.meas.get_counts().items()
    }
    for res in result
]
plot_histogram(
    binary_prob + [ideal_distribution],
    bar_labels=False,
    legend=[
        "optimization_level=0",
        "optimization_level=3",
        "optimization_level=3 + dd",
        "ideal distribution",
    ],
)

<Image src="../docs/images/guides/circuit-transpilation-settings/extracted-outputs/525777ea-d438-4f3b-acb6-53e579f24a0e-0.avif" alt="Output of the previous code cell" />

![Rezultatul celulei de cod anterioare](../docs/images/guides/circuit-transpilation-settings/extracted-outputs/4ada6498-b9d7-4d88-b8a9-ef1dc0a85bf7-0.avif)

## Pasul 3: Execută folosind primitivele Qiskit
În acest moment, ai o listă de circuite transpilate pentru QPU-ul specificat. Apoi, creează o instanță a primitivei Sampler și pornește un job în lot folosind managerul de context (`with ...:`), care deschide și închide automat lotul.

În interiorul managerului de context, eșantionează circuitele și stochează rezultatele în `result`.

In [ ]:
for prob in binary_prob:
    print(f"{hellinger_fidelity(prob, ideal_distribution):.3f}")

0.848
0.945
0.990
